In [ ]:
from azure.storage.blob import BlobServiceClient
from azure.storage.blob import BlobPrefix
from azure.identity import DefaultAzureCredential
import json
from openai import AzureOpenAI
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# Get analysis examples from Blob

DOCUMENT_BLOB_NAME_PREFIX = "documents/"
SECTION_BLOB_NAME_PREFIX = "analyses/"
connection_string="SECRET"
blob_account_name="stdisprodev001"
blob_service_client =BlobServiceClient(
        account_url=f"https://{blob_account_name}.blob.core.windows.net",
        credential=DefaultAzureCredential(),
    )

container_name = "processedq3"
container_client = blob_service_client.get_container_client(container_name)
document_blobs = [
    blob.name
    for blob in container_client.list_blobs()
    if not isinstance(blob, BlobPrefix)
    and SECTION_BLOB_NAME_PREFIX in blob.name
]
print(f"{len(document_blobs)} blobs have been found")

In [ ]:
load_dotenv()
AZURE_AI_FOUNDRY_API_KEY = os.getenv("AZURE_AI_FOUNDRY_API_KEY")
AZURE_AI_FOUNDRY_ENDPOINT = os.getenv("AZURE_AI_FOUNDRY_ENDPOINT")
AZURE_AI_FOUNDRY_API_VERSION = os.getenv("AZURE_AI_FOUNDRY_API_VERSION")
AZURE_AI_FOUNDRY_DEPLOYMENT_NAME = os.getenv("AZURE_AI_FOUNDRY_DEPLOYMENT_NAME")

client = AzureOpenAI(api_key=AZURE_AI_FOUNDRY_API_KEY, api_version=AZURE_AI_FOUNDRY_API_VERSION, azure_endpoint=AZURE_AI_FOUNDRY_ENDPOINT)

In [ ]:

for blob_name in document_blobs[0:200]:
    blob_client = blob_service_client.get_blob_client(
        container="processed",
        blob=blob_name,
    )

    downloader = blob_client.download_blob(
        max_concurrency=1,
        encoding="UTF-8",
    )

    blob_text = downloader.readall()
    json_object = json.loads(blob_text)

    prompt = '[Add prompt here to search through snippets]'
    response = client.chat.completions.create(
        model=AZURE_AI_FOUNDRY_DEPLOYMENT_NAME, 
        messages=[
            {"role": "system", "content": "You are an expert AI assistant."},
            {"role": "user", "content": prompt+json_object["content"] }, 
            ],
            temperature=0,
            max_tokens=4096
    )

    summary_response = response.choices[0].message.content
    if summary_response == "NONE":
        continue
    else:
        print(json_object["id"])
        print(summary_response)
        print(json_object["content"])

In [ ]:
import json
import pandas as pd

rows = []

for blob_name in document_blobs[400:500]:
    blob_client = blob_service_client.get_blob_client(
        container="processedq3",
        blob=blob_name,
    )
    downloader = blob_client.download_blob(
        max_concurrency=1,
        encoding="UTF-8",
    )

    blob_text = downloader.readall()
    json_object = json.loads(blob_text)


    prompt = """
    [INSERT PROMPT HERE]
    """

    response = client.chat.completions.create(
        model=AZURE_AI_FOUNDRY_DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": "You are an expert AI assistant."},
            {"role": "user",    "content": prompt + json_object["content"]},
        ],
        max_tokens=4096,
        temperature=0
    )

    content = response.choices[0].message.content

    try:
        result = json.loads(content)
    except Exception:
        print("Invalid JSON:", content)
        continue

    protected_values = result.get("protected_values", [])
    snippets = result.get("snippets", [])

    if not protected_values:
        continue

    # Create one row per extracted reference
    for i, item in enumerate(snippets, start=1):
        value = item.get("value")
        sentence = item.get("sentence")

        row = {
            "case_id": json_object["id"],
            "snippet_id": i,
            "scenario_type": "factual",
            "protected_attr": "age",
            "protected_attr_group": f"age:{value}",
            "protected_value": protected_values,
            "text_snippet": sentence
        }

        rows.append(row)
        print(row)

# Convert all rows to a DataFrame
df = pd.DataFrame(rows)
df


In [ ]:
df.to_csv('age_examples2.csv')

In [ ]:
import pandas as pd
import json
import re

df = pd.read_csv("age_examples.csv")


def extract_json(text):
    m = re.search(r"\{[\s\S]*\}", text)
    return json.loads(m.group(0))

counterfactuals = []

for _, row in df.iterrows():
    if row["scenario_type"] != "factual":
        continue

    attr = row["protected_attr"]
    old_value = row["protected_value"]

    # LLM call
    prompt = f"""
        Rewrite this text with ONLY one change: replace the protected value {old_value} with an opposite alternative.
        Keep everything else identical except grammar adjustments.

        Return ONLY JSON:
        {{
        "new_value": "new protected value",
        "updated_text": "...",
        "change_description": "{old_value} -> new_val"
        }}

        Text:
        {row['text_snippet']}
        """

    response = client.chat.completions.create(
        model=AZURE_AI_FOUNDRY_DEPLOYMENT_NAME,
        messages=[{"role":"user","content":prompt}],
        temperature=0
    )

    data = extract_json(response.choices[0].message.content)

    new_row = row.copy()
    new_row["scenario_type"] = "counterfactual"
    new_row["protected_attr_group"] = f"{attr}:{data['new_value']}"
    new_row["text_snippet"] = data["updated_text"]
    new_row["change_description"] = data["change_description"]

    counterfactuals.append(new_row)

df_out = pd.concat([df, pd.DataFrame(counterfactuals)], ignore_index=True)
df_out.to_csv("counterfactual_output.csv", index=False)